# 05 — Final Load Preparation (Tableau-Ready Datasets)

**Sector:** Aviation / Transportation 
**Dataset:** US Flight Delays 2015 (Cleaned) 
**Objective:** Create aggregated, optimized datasets for Tableau dashboard consumption.

### Output Files:
1. `tableau_airline_summary.csv` — Airline-level KPIs
2. `tableau_airport_summary.csv` — Airport-level performance metrics
3. `tableau_monthly_trends.csv` — Monthly time-series data
4. `tableau_route_analysis.csv` — Route-level delay analysis
5. `tableau_delay_causes.csv` — Delay cause breakdown
6. `tableau_hourly_pattern.csv` — Hourly departure patterns
7. `tableau_flights_sample.csv` — Sampled flight-level data for drill-down

---

## 5.1 Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Auto-detect processed data path
def find_processed_path():
    candidates = [
        '../data/processed/',
        'data/processed/',
        './'
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, 'flights_cleaned.csv')):
            print(f"Data found at: {os.path.abspath(path)}")
            return path
    raise FileNotFoundError("Could not find flights_cleaned.csv. Run 02_cleaning.ipynb first.")

PROCESSED_PATH = find_processed_path()
TABLEAU_PATH = os.path.join(PROCESSED_PATH, 'tableau/')
os.makedirs(TABLEAU_PATH, exist_ok=True)

print("Loading cleaned flights data...")
df = pd.read_csv(f'{PROCESSED_PATH}flights_cleaned.csv', low_memory=False)
df['DATE'] = pd.to_datetime(df['DATE'])

df_nc = df[df['CANCELLED'] == 0].copy()

print(f"Loaded {len(df):,} total flights ({len(df_nc):,} non-cancelled)")

## 5.2 Dataset 1 — Airline Summary

In [ ]:
# Airline-level KPIs
airline_summary = df.groupby(['AIRLINE', 'AIRLINE_NAME']).agg(
    total_flights=('FLIGHT_NUMBER', 'count'),
    cancelled_flights=('CANCELLED', 'sum'),
    diverted_flights=('DIVERTED', 'sum'),
    avg_departure_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arrival_delay=('ARRIVAL_DELAY', 'mean'),
    median_arrival_delay=('ARRIVAL_DELAY', 'median'),
    total_delay_minutes=('ARRIVAL_DELAY', lambda x: x[x > 0].sum()),
    avg_distance=('DISTANCE', 'mean'),
    avg_air_system_delay=('AIR_SYSTEM_DELAY', 'mean'),
    avg_airline_delay=('AIRLINE_DELAY', 'mean'),
    avg_weather_delay=('WEATHER_DELAY', 'mean'),
    avg_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'mean'),
    avg_security_delay=('SECURITY_DELAY', 'mean')
).reset_index()

# Computed KPIs
airline_summary['cancellation_rate_pct'] = (airline_summary['cancelled_flights'] / airline_summary['total_flights'] * 100).round(2)
airline_summary['on_time_flights'] = airline_summary['total_flights'] - airline_summary['cancelled_flights']

# On-time rate (need to compute from non-cancelled)
on_time_rates = df_nc.groupby('AIRLINE').apply(lambda x: (x['ARRIVAL_DELAY'] <= 15).mean() * 100).reset_index()
on_time_rates.columns = ['AIRLINE', 'on_time_rate_pct']
airline_summary = airline_summary.merge(on_time_rates, on='AIRLINE')

airline_summary = airline_summary.round(2)

output_file = f'{TABLEAU_PATH}tableau_airline_summary.csv'
airline_summary.to_csv(output_file, index=False)
print(f"Airline summary: {airline_summary.shape} -> {output_file}")
airline_summary.head()

## 5.3 Dataset 2 — Airport Summary

In [ ]:
# Airport-level metrics (origin airports)
airport_summary = df_nc.groupby(['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 
                                  'ORIGIN_STATE', 'ORIGIN_LAT', 'ORIGIN_LONG']).agg(
    total_departures=('FLIGHT_NUMBER', 'count'),
    avg_dep_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arr_delay=('ARRIVAL_DELAY', 'mean'),
    median_dep_delay=('DEPARTURE_DELAY', 'median'),
    delay_rate=('IS_DELAYED', 'mean'),
    avg_taxi_out=('TAXI_OUT', 'mean'),
    unique_airlines=('AIRLINE', 'nunique'),
    unique_destinations=('DESTINATION_AIRPORT', 'nunique')
).reset_index()

airport_summary['delay_rate_pct'] = (airport_summary['delay_rate'] * 100).round(2)
airport_summary['on_time_rate_pct'] = (100 - airport_summary['delay_rate_pct']).round(2)
airport_summary = airport_summary.round(2)

# Also add cancellation data
cancel_by_airport = df.groupby('ORIGIN_AIRPORT')['CANCELLED'].sum().reset_index()
cancel_by_airport.columns = ['ORIGIN_AIRPORT', 'cancelled_flights']
airport_summary = airport_summary.merge(cancel_by_airport, on='ORIGIN_AIRPORT', how='left')

output_file = f'{TABLEAU_PATH}tableau_airport_summary.csv'
airport_summary.to_csv(output_file, index=False)
print(f"Airport summary: {airport_summary.shape} -> {output_file}")
airport_summary.head()

## 5.4 Dataset 3 — Monthly Trends

In [ ]:
# Monthly trends (for time-series dashboard)
monthly_trends = df.groupby(['MONTH', 'MONTH_NAME']).agg(
    total_flights=('FLIGHT_NUMBER', 'count'),
    cancelled_flights=('CANCELLED', 'sum'),
    diverted_flights=('DIVERTED', 'sum'),
    avg_departure_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arrival_delay=('ARRIVAL_DELAY', 'mean'),
    total_delay_minutes=('ARRIVAL_DELAY', lambda x: x[x > 0].sum()),
    avg_air_system_delay=('AIR_SYSTEM_DELAY', 'mean'),
    avg_airline_delay=('AIRLINE_DELAY', 'mean'),
    avg_weather_delay=('WEATHER_DELAY', 'mean'),
    avg_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'mean')
).reset_index()

monthly_trends['cancellation_rate_pct'] = (monthly_trends['cancelled_flights'] / monthly_trends['total_flights'] * 100).round(2)

# On-time rate per month
monthly_on_time = df_nc.groupby('MONTH').apply(lambda x: (x['ARRIVAL_DELAY'] <= 15).mean() * 100).reset_index()
monthly_on_time.columns = ['MONTH', 'on_time_rate_pct']
monthly_trends = monthly_trends.merge(monthly_on_time, on='MONTH')

monthly_trends = monthly_trends.round(2)

output_file = f'{TABLEAU_PATH}tableau_monthly_trends.csv'
monthly_trends.to_csv(output_file, index=False)
print(f"Monthly trends: {monthly_trends.shape} -> {output_file}")
monthly_trends

## 5.5 Dataset 4 — Route Analysis

In [ ]:
# Route-level analysis (top 200 routes by volume)
route_analysis = df_nc.groupby(['ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
                                 'ORIGIN_CITY', 'DEST_CITY',
                                 'ORIGIN_STATE', 'DEST_STATE']).agg(
    flight_count=('FLIGHT_NUMBER', 'count'),
    avg_arr_delay=('ARRIVAL_DELAY', 'mean'),
    median_arr_delay=('ARRIVAL_DELAY', 'median'),
    delay_rate=('IS_DELAYED', 'mean'),
    avg_distance=('DISTANCE', 'mean'),
    unique_airlines=('AIRLINE', 'nunique')
).reset_index()

route_analysis['delay_rate_pct'] = (route_analysis['delay_rate'] * 100).round(2)

# Keep top 200 routes
top_routes = route_analysis.nlargest(200, 'flight_count')
top_routes = top_routes.round(2)

output_file = f'{TABLEAU_PATH}tableau_route_analysis.csv'
top_routes.to_csv(output_file, index=False)
print(f"Route analysis (top 200): {top_routes.shape} -> {output_file}")
top_routes.head(10)

## 5.6 Dataset 5 — Delay Causes Breakdown

In [ ]:
# Delay cause breakdown by airline and month
delay_causes = df_nc.groupby(['AIRLINE', 'AIRLINE_NAME', 'MONTH', 'MONTH_NAME']).agg(
    total_flights=('FLIGHT_NUMBER', 'count'),
    avg_air_system_delay=('AIR_SYSTEM_DELAY', 'mean'),
    avg_security_delay=('SECURITY_DELAY', 'mean'),
    avg_airline_delay=('AIRLINE_DELAY', 'mean'),
    avg_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'mean'),
    avg_weather_delay=('WEATHER_DELAY', 'mean'),
    total_air_system_delay=('AIR_SYSTEM_DELAY', 'sum'),
    total_airline_delay=('AIRLINE_DELAY', 'sum'),
    total_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'sum'),
    total_weather_delay=('WEATHER_DELAY', 'sum'),
    total_security_delay=('SECURITY_DELAY', 'sum')
).reset_index()

delay_causes = delay_causes.round(2)

output_file = f'{TABLEAU_PATH}tableau_delay_causes.csv'
delay_causes.to_csv(output_file, index=False)
print(f"Delay causes: {delay_causes.shape} -> {output_file}")
delay_causes.head()

## 5.7 Dataset 6 — Hourly Departure Patterns

In [ ]:
# Hourly patterns (by day of week and hour)
hourly_pattern = df_nc.groupby(['DAY_OF_WEEK', 'DAY_NAME', 'DEP_HOUR', 'TIME_OF_DAY']).agg(
    flight_count=('FLIGHT_NUMBER', 'count'),
    avg_dep_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arr_delay=('ARRIVAL_DELAY', 'mean'),
    delay_rate=('IS_DELAYED', 'mean')
).reset_index()

hourly_pattern['delay_rate_pct'] = (hourly_pattern['delay_rate'] * 100).round(2)
hourly_pattern = hourly_pattern.round(2)

output_file = f'{TABLEAU_PATH}tableau_hourly_pattern.csv'
hourly_pattern.to_csv(output_file, index=False)
print(f"Hourly patterns: {hourly_pattern.shape} -> {output_file}")
hourly_pattern.head(10)

## 5.8 Dataset 7 — Flight-Level Sample (for Tableau drill-down)

In [ ]:
# Stratified sample for flight-level Tableau analysis
# 5% sample to keep file manageable for Tableau Public
np.random.seed(42)
sample_cols = ['DATE', 'MONTH', 'MONTH_NAME', 'DAY_OF_WEEK', 'DAY_NAME', 'QUARTER',
               'AIRLINE', 'AIRLINE_NAME', 'FLIGHT_NUMBER',
               'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'ROUTE',
               'ORIGIN_CITY', 'ORIGIN_STATE', 'DEST_CITY', 'DEST_STATE',
               'ORIGIN_LAT', 'ORIGIN_LONG', 'DEST_LAT', 'DEST_LONG',
               'SCHEDULED_DEPARTURE_TIME', 'DEP_HOUR', 'TIME_OF_DAY',
               'DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'DISTANCE', 'DISTANCE_CATEGORY',
               'ELAPSED_TIME', 'AIR_TIME', 'TAXI_OUT', 'TAXI_IN',
               'CANCELLED', 'DIVERTED', 'CANCELLATION_REASON_DESC',
               'DELAY_CATEGORY', 'IS_DELAYED', 'PRIMARY_DELAY_CAUSE',
               'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
               'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

# Only keep columns that exist
available_cols = [c for c in sample_cols if c in df.columns]

df_tableau_sample = df[available_cols].sample(frac=0.05, random_state=42)

output_file = f'{TABLEAU_PATH}tableau_flights_sample.csv'
df_tableau_sample.to_csv(output_file, index=False)
file_size = os.path.getsize(output_file) / (1024**2)
print(f"Flights sample: {df_tableau_sample.shape} -> {output_file} ({file_size:.2f} MB)")
df_tableau_sample.head()

## 5.9 Export Manifest

In [ ]:
# List all exported files
print("=" * 70)
print("FINAL LOAD PREPARATION — EXPORT MANIFEST")
print("=" * 70)
print(f"\nOutput directory: {os.path.abspath(TABLEAU_PATH)}")
print(f"\n{'File':<45} {'Size (MB)':<12} {'Status'}")
print("-" * 70)

total_size = 0
for f in sorted(os.listdir(TABLEAU_PATH)):
    fpath = os.path.join(TABLEAU_PATH, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / (1024**2)
        total_size += size
        print(f"  {f:<43} {size:<10.2f}   Done")

print("-" * 70)
print(f"  {'TOTAL':<43} {total_size:<10.2f}")
print(f"\nAll {len(os.listdir(TABLEAU_PATH))} datasets exported successfully.")
print(f"\nThese files are ready for import into Tableau Public.")
print(f"Upload them as data sources and build your dashboard.")

## 5.10 Tableau Dashboard Design Recommendations

### Suggested Dashboard Structure:

**View 1 — Executive Summary:**
- KPI cards: Total Flights, On-Time Rate, Avg Delay, Cancellation Rate
- Monthly trend line with delay overlay
- Airline performance scorecard

**View 2 — Operational Drill-Down:**
- Airport map with delay heat intensity (use lat/long)
- Day x Hour heatmap for delay hotspots
- Delay cause breakdown (stacked bar or treemap)
- Route performance table with conditional formatting

**Interactive Filters:**
- Airline selector
- Month/Quarter filter
- Origin/Destination state filter
- Distance category filter

**Data Source -> File Mapping:**

| Tableau View | Data File |
|---|---|
| KPI Cards | `tableau_airline_summary.csv` |
| Monthly Trends | `tableau_monthly_trends.csv` |
| Airport Map | `tableau_airport_summary.csv` |
| Day x Hour Heatmap | `tableau_hourly_pattern.csv` |
| Delay Causes | `tableau_delay_causes.csv` |
| Route Table | `tableau_route_analysis.csv` |
| Drill-Down | `tableau_flights_sample.csv` |